# A Simple Calculator Using EBNF

This file shows the implementation of a *symbolic calculator* using `lark`.  It is a variant of the notebook
`01-Calculator.ipynb`:  Instead of left recursion, the grammar uses the repetition operator `*` of the
*extended Backus-Naur form* (EBNF).  The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{stmnt}   & \rightarrow & \;\texttt{IDENTIFIER}\; \texttt{':='} \; \;\texttt{expr}\; \texttt{';'} \\
                   & \mid        & \;\texttt{expr}\; \texttt{';'} \\[0.2cm]
  \texttt{expr}    & \rightarrow & \;\texttt{product}\; \bigl((\texttt{'+'} \mid \texttt{'-'})\; \texttt{product}\bigr)^* \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{factor}\; \bigl((\texttt{'*'} \mid \texttt{'/'})\; \texttt{factor}\bigr)^* \\[0.2cm]
  \texttt{factor}  & \rightarrow &   \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \;\texttt{NUMBER}                                   \\
                   & \mid        & \;\texttt{IDENTIFIER}
\end{array}
$$
The abstract syntax tree and its evaluation are the same as in `01-Calculator.ipynb`.

## Setup and Imports
We import `Lark` for parsing and `Token` to identify leaf nodes when walking the syntax tree.

In [ ]:
from lark import Lark, Token

## Specification of the Grammar

Unlike separated lexer and parser libraries, `lark` allows us to define both lexical tokens and structural rules natively in a single EBNF string.
Let us break down the components of our grammar definition:

* `?start: stmnt`: This tells Lark that the entry point for our grammar is a statement (`stmnt`).
   The `?` prefix instructs the parser to inline this rule if it only has one child, preventing unnecessary nesting in our syntax tree.
* `stmnt`: A statement is either an assignment or an expression followed by a semicolon.  The **aliases** `-> assign` and
   `-> expr_stmnt` give the nodes of the two alternatives different names, so that we can tell them apart when we build our
   Abstract Syntax Tree (AST).
* **Repetition** `( ... )*`: The rule `?expr: product (ADD_OP product)*` states that an expression is a product that is followed
   by zero or more occurrences of an additive operator and another product.  This single rule replaces the three left-recursive rules
   ```
   ?expr: expr "+" product -> add
        | expr "-" product -> sub
        | product
   ```
   of `01-Calculator.ipynb`.  The rule for `product` is built the same way.  The hierarchy `expr`, `product`, `factor` still
   enforces operator precedence (e.g., multiplication in `product` happens before addition in `expr`).
* **Flat nodes**: If the repetition occurs zero times, the node has only one child and the `?` prefix inlines it.  Otherwise, a single
   node `expr` (or `product`) is created whose children are alternately operands and operators.  For example, the expression
   `1 - 2 - 3` yields one node `expr` with five children.  Hence the grammar no longer specifies that `-` associates to the left.
   We take care of this when we build the AST.
* **Named Terminals** (`ADD_OP`, `MUL_OP`): Lark removes anonymous strings like `"+"` from the parse tree.  Since we need to know
   which operator has been used, we give the operators a name: Named terminals are kept in the tree.  Note that the definition of a
   terminal may use the alternative `|`, too.
* **Lexical Imports** (`%import common...`): Instead of manually writing regular expressions for common tokens, we import standard terminal
   definitions provided by Lark.
   * `CNAME` parses standard variable names (letters, numbers, underscores) and is aliased to `IDENTIFIER`.
   * `NUMBER` handles both integers and floating-point numbers.
   * `WS` handles whitespace.
* `ASSIGN` and `SEMI` give readable names to the tokens `":="` and `";"`.  As they are written as anonymous strings in the rule for
   `stmnt`, they are still removed from the parse tree.
* `%ignore WS`: This directs the lexer to completely discard spaces and tabs so they do not clutter our parsing logic.

In [ ]:
calc_grammar = """
?start: stmnt

?stmnt: IDENTIFIER ":=" expr ";" -> assign
      | expr ";"                 -> expr_stmnt

?expr: product (ADD_OP product)*

?product: factor (MUL_OP factor)*

?factor: "(" expr ")"
       | NUMBER                  -> number
       | IDENTIFIER              -> var

// Lexical definitions
%import common.CNAME -> IDENTIFIER
%import common.NUMBER
%import common.WS

ADD_OP: "+" | "-"
MUL_OP: "*" | "/"
ASSIGN: ":="
SEMI:   ";"

// Ignore whitespace
%ignore WS
"""

Instantiate the parser using the LALR algorithm. We do not use a Transformer here.

In [ ]:
parser = Lark(calc_grammar, parser='lalr')

The parse tree of `1 - 2 * 3 - 4;` shows the flat nodes:  The children of the node `expr` are alternately operands and operators.

In [ ]:
print(parser.parse('1 - 2 * 3 - 4;').pretty())

## Generating the Abstract Syntax Tree (ADT)

Rather than relying on library-specific objects to evaluate our code, we will manually walk the `lark.Tree` generated by the parser and convert it into a pure Python Abstract Data Type (ADT) represented by nested tuples.

For example, the string `x := y + z;` will be parsed into the ADT `(':=', 'x', ('+', ('var', 'y'), ('var', 'z')))`.

The children of a flat node `expr` or `product` have the form `[operand, op, operand, op, ..., operand]`.  We combine them
from left to right, so `1 - 2 - 3` becomes `('-', ('-', ('num', '1'), ('num', '2')), ('num', '3'))`.  The operators are tokens and
are converted into the strings `'+'`, `'-'`, `'*'`, and `'/'`, which are exactly the operator names of our ADT.  Therefore, the ADT
is the same as in `01-Calculator.ipynb`.

In [ ]:
def ast_to_tuples(node):
    # Base case: If it's a leaf node (a Token), return its string value
    if isinstance(node, Token):
        return str(node)
    # Recursively convert all children
    children = [ast_to_tuples(child) for child in node.children]
    # Combine the operands of a flat node from left to right
    if node.data in ('expr', 'product'):
        result = children[0]
        for op, operand in zip(children[1::2], children[2::2]):
            result = (op, result, operand)
        return result
    # Map Lark's node names to our preferred operator strings
    operator_map = {
        'assign':     ':=',
        'expr_stmnt': 'eval',
        'number':     'num',
        'var':        'var'
    }
    # Get the operator string (default to the raw node data if not in the map)
    op = operator_map.get(node.data, node.data)
    # Return the nested tuple
    return (op, *children)

## Evaluating the Abstract Syntax Tree

Now that we have our pure ADT made of nested tuples, we can write our own recursive evaluation function. The variable `Names2Values` is a global dictionary that stores the values associated with the variables we define.

In [ ]:
Names2Values = {}

In [ ]:
def evaluate_adt(adt):
    match adt:
        case ('num', value):
            return float(value)
        case ('var', var_name):
            if var_name not in Names2Values:
                print(f"Syntax Error: Variable '{var_name}' is undefined.")
                return 0.0
            return Names2Values[var_name]
        case (':=', var_name, expr):
            Names2Values[var_name] = evaluate_adt(expr)         
        case ('eval', expr):
            print(evaluate_adt(expr))
        case (('+' | '-' | '*' | '/') as op, left, right):
            val_left  = evaluate_adt(left )
            val_right = evaluate_adt(right)
            match op:
                case '+': return val_left + val_right
                case '-': return val_left - val_right
                case '*': return val_left * val_right
                case '/': return val_left / val_right
        case _:
            raise ValueError(f"Unknown ADT structure: {adt}")

## Execution

The REPL (Read-Eval-Print Loop) takes the input string, parses it into Lark's tree, converts it into our nested tuples, and then evaluates it.

In [ ]:
def main():
    while True:
        try:
            s = input('calc> ')
            if s == '':
                break
            # 1. Parse the string into the Lark Tree
            raw_tree = parser.parse(s)
            # 2. Convert it to a nested tuple
            adt_tuples = ast_to_tuples(raw_tree)           
            # 3. Evaluate the nested tuple 
            evaluate_adt(adt_tuples)
        except (EOFError, NotImplementedError):  # no input possible
            break
        except Exception as e:
            print(f"Error: {e}")

In [ ]:
main()

Check the populated memory map:

In [ ]:
Names2Values

We can even get the parser states.  As LALR parsing works with plain BNF rules, Lark translates the repetition
`(ADD_OP product)*` into the auxiliary left-recursive rules
```
__expr_star_0: ADD_OP product
             | __expr_star_0 ADD_OP product
```
and `expr: product __expr_star_0 | product`, which show up in the states.  Since the name `__expr_star_0` starts with an underscore,
its nodes are inlined into their parent.  This is why the node `expr` is flat.

In [ ]:
from lark import Lark
from lark.parsers.lalr_analysis import LALR_Analyzer

In [ ]:
# 1. Load the autoreload extension so local file changes are instantly recognized
%load_ext autoreload
%autoreload 2

In [ ]:
# 2. Import your custom library
from formal_methods_viz import print_lalr1_states

In [ ]:
# 3. Define the token map for the grammar above
token_map = {
    'ASSIGN': "':='",
    'SEMI': "';'",
    'IDENTIFIER': "'id'",
    'NUMBER': "'num'",

    # Mathematical Operators
    'ADD_OP': "'addop'",
    'MUL_OP': "'mulop'",

    # Parentheses
    'LPAR': "'('",
    'RPAR': "')'"
}

In [ ]:
# 4. Generate the states!
print_lalr1_states(parser, token_map)